#### Stream Customers data from Cloud files to Delta Lake using Auto Loader
    1. Read files from Cloud storgae using Auto Loader
    2. Transform the dataframe to add the following columns
            filepath: Cloud file path
            ingestiondate: Current Timestamp
    3. Write the transformed data stream to Delata Lake Table

##### 1. Read files using Auto Loader

In [0]:
customers_df = (
                    spark.readStream
                        .format("cloudFiles")
                        .option("cloudFiles.format", "json")
                        .option("cloudFiles.schemaLocation", "/Volumes/gizmobox_catalog_subbu/landing/operational_data_volume/customers_autoloader/_schema")
                        .option("cloudFiles.inferColumnTypes", "true")   
                        .option("cloudFiles.schemaHints", "date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP")
                        .option("pathGlobFilter", "customers_2024_*.json")                     
                        .load("/Volumes/gizmobox_catalog_subbu/landing/operational_data_volume/customers_autoloader/")
)

##### 2.Transform the dataframe to add the following columns
-  filepath: Cloud file path
-  ingestiondate: Current Timestamp



In [0]:
from pyspark.sql.functions import col, current_timestamp
customers_transformed_df = (
                            customers_df
                                .withColumn("file_path", col("_metadata.file_path"))
                                .withColumn("ingestion_date", current_timestamp())
)

##### 3. Write the transformed data stream to Delata Lake Table

In [0]:
streaming_query = ( customers_transformed_df.writeStream
                        .format("delta")
                        # .outputMode("append")
                        # .trigger(processingTime="60 seconds")
                        .option("checkpointLocation", "/Volumes/gizmobox_catalog_subbu/landing/operational_data_volume/customers_autoloader/_checkpoint_stream")
                        .toTable("gizmobox_catalog_subbu.bronze.customers_autoloader")
)

In [0]:
streaming_query.stop()

In [0]:
%sql
DROP TABLE gizmobox_catalog_subbu.bronze.customers_autoloader

In [0]:
%sql
select * from gizmobox_catalog_subbu.bronze.customers_autoloader